## ECG Heartbeat Categorization

> This dataset is composed of two collections of heartbeat signals derived from two famous datasets in heartbeat classification, the MIT-BIH Arrhythmia Dataset and The PTB Diagnostic ECG Database.

In [ ]:
# importing necessary libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
class_names = ['0: Normal', '1: PAC', '2: PVC', '3: Fusion', '4: Paced/Unknown']


In [ ]:
# importing dataset with multi-path resolution (works from workspace root, notebooks/, or Kaggle)
import os
train_candidates = [
    'data/mitbih_train.csv',
    '../data/mitbih_train.csv',
    'input/heartbeat/mitbih_train.csv',
    '../input/heartbeat/mitbih_train.csv'
]
train_path = next((p for p in train_candidates if os.path.exists(p)), 'data/mitbih_train.csv')
data = pd.read_csv(train_path, header=None)
df = pd.DataFrame(data)
print(f'Successfully loaded dataset from: {train_path}')
print(f'Dataset Shape: {df.shape} (Heartbeats: {df.shape[0]}, Features: {df.shape[1]-1} + 1 Class Label)')


In [ ]:
df.head()

In [ ]:
# showing column wise percentage of NaN values they contain
total_nulls = df.isna().sum().sum()
print(f'Total missing values in dataset: {total_nulls}')
if total_nulls == 0:
    print('Data is completely clean with 0 null values!')


> Since data does'nt contain any null values, we can move further

In [ ]:
# Imbalanced class distribution visualization (Fixed for Seaborn 0.13+)
plt.figure(figsize=(9, 4))
sns.countplot(x=df[187], hue=df[187], palette='viridis', legend=False)
plt.title('Imbalanced Heartbeat Class Distribution (Original Dataset)', fontsize=13, fontweight='bold')
plt.xlabel('Heartbeat Class Label', fontsize=11)
plt.ylabel('Number of Samples', fontsize=11)
plt.xticks(ticks=range(5), labels=class_names)
plt.grid(True, alpha=0.3)
plt.show()


> Here this bar graph easily shows how data is imbalanced. More than 80% data is in class 0. So, first, we have to balance th data in to get more precise predictions.

> For balancing the data I'm using undersampling in which we will reduce the rows of class 0 to the number compareble to others/

In [ ]:
# Class-wise separation and undersampling
class_1 = df[df[187] == 1.0]
class_2 = df[df[187] == 2.0]
class_3 = df[df[187] == 3.0]
class_4 = df[df[187] == 4.0]

# Sample class 0 safely without exceeding population
sample_n = min(len(df[df[187] == 0.0]), 8000)
class_0 = df[df[187] == 0.0].sample(n=sample_n, random_state=42)
print(f'Sampled {sample_n} heartbeats from Class 0.')


In [ ]:
new_df = pd.concat([class_0, class_1, class_2, class_3, class_4])

In [ ]:
new_df.head()

In [ ]:
# Visualizing balanced class distribution
plt.figure(figsize=(9, 4))
sns.countplot(x=new_df[187], hue=new_df[187], palette='Set2', legend=False)
plt.title('Balanced Heartbeat Class Distribution (After Undersampling)', fontsize=13, fontweight='bold')
plt.xlabel('Heartbeat Class Label', fontsize=11)
plt.ylabel('Number of Samples', fontsize=11)
plt.xticks(ticks=range(5), labels=class_names)
plt.grid(True, alpha=0.3)
plt.show()


> Now for visualising each class, here is plot of any random sample of hearbeat in each class.

In [ ]:
# Plot random sample waveform for each heartbeat class (Full 187 sample points)
fig, ax = plt.subplots(nrows=1, ncols=5, figsize=(25, 3))
colors = ['#10b981', '#3b82f6', '#ef4444', '#f59e0b', '#8b5cf6']

for i in range(5):
    sample_df = new_df[new_df[187] == float(i)]
    if len(sample_df) > 0:
        # Columns 0 to 186 contain the 187 waveform samples
        waveform = sample_df.sample(1, random_state=42).iloc[0, :187].values
        ax[i].plot(waveform, color=colors[i], linewidth=2)
        ax[i].set_title(class_names[i], fontsize=12, fontweight='bold')
        ax[i].set_xlabel('Sample Index (0-186)')
        ax[i].set_ylabel('Normalized Voltage')
        ax[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Split data into train and test sets with stratification
from sklearn.model_selection import train_test_split

X = new_df.drop([187], axis=1).values
y = new_df[187].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')


In [ ]:
# Reshape for 1D Convolutional Neural Network: (samples, channels, length)
X_train = np.array(X_train).reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = np.array(X_test).reshape(X_test.shape[0], 1, X_test.shape[1])
print(f'X_train reshaped for 1D-CNN: {X_train.shape}')
print(f'X_test reshaped for 1D-CNN:  {X_test.shape}')


In [ ]:
# 1D-CNN Engine: PyTorch with native Python 3.14 Windows support
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')


In [ ]:
# Define 1D-CNN Architecture
# Conv1D(32, k=3) -> Conv1D(64, k=3) -> Conv1D(128, k=5) -> MaxPool(k=3, s=2) -> Dropout(0.5) -> Dense(512) -> Dense(1024) -> Dense(5)
class ECG_CNN1D(nn.Module):
    def __init__(self, num_classes=5):
        super(ECG_CNN1D, self).__init__()
        # Conv1: 32 filters, kernel 3, padding 'same' (p=1)
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        # Conv2: 64 filters, kernel 3, padding 'same' (p=1)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        # Conv3: 128 filters, kernel 5, padding 'same' (p=2)
        self.conv3 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=5, padding=2)
        self.relu3 = nn.ReLU()
        # MaxPool with stride 2 and padding 1
        self.pool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        self.dropout = nn.Dropout(0.5)
        # Linear layers (187 pooled with stride 2 -> 94 length -> 128*94 = 12032)
        self.fc1 = nn.Linear(128 * 94, 512)
        self.relu_fc1 = nn.ReLU()
        self.fc2 = nn.Linear(512, 1024)
        self.relu_fc2 = nn.ReLU()
        self.out = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.relu2(self.conv2(x))
        x = self.relu3(self.conv3(x))
        x = self.pool(x)
        x = self.dropout(x)
        x = x.view(x.size(0), -1)
        x = self.relu_fc1(self.fc1(x))
        x = self.relu_fc2(self.fc2(x))
        x = self.out(x)
        return x

clf = ECG_CNN1D(num_classes=5).to(device)
print(clf)


In [ ]:
# Loss function and Adam Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf.parameters(), lr=0.001)


In [ ]:
# Prepare PyTorch DataLoaders & Train for 10 Epochs
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=128, shuffle=False)

epochs = 10
history_acc = []
history_loss = []

print('Starting 1D-CNN Training...')
for epoch in range(1, epochs + 1):
    clf.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = clf(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch_x.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
        
    epoch_acc = correct / total
    epoch_loss = total_loss / total
    history_acc.append(epoch_acc)
    history_loss.append(epoch_loss)
    print(f'Epoch [{epoch:02d}/{epochs:02d}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc*100:.2f}%')

print('Training complete!')


In [ ]:
# Validation Inference
clf.eval()
val_preds = []
with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        outputs = clf(batch_x)
        _, preds = torch.max(outputs, 1)
        val_preds.extend(preds.cpu().numpy())

y_lbl = np.array(val_preds)


In [ ]:
# Plot Training Accuracy & Loss History (Fixed Y-axis labels)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
epochs_range = range(1, len(history_acc) + 1)

# Accuracy subplot
ax[0].plot(epochs_range, history_acc, color='#3b82f6', marker='o', linewidth=2, label='Training Accuracy')
ax[0].set_title('Training Accuracy Curve', fontsize=12, fontweight='bold')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Accuracy')
ax[0].grid(True, alpha=0.3)
ax[0].legend()

# Loss subplot
ax[1].plot(epochs_range, history_loss, color='#ef4444', marker='s', linewidth=2, label='Training Loss')
ax[1].set_title('Training Loss Curve', fontsize=12, fontweight='bold')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Cross-Entropy Loss')
ax[1].grid(True, alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Validation Confusion Matrix Heatmap
mat = confusion_matrix(y_test, y_lbl)
plt.figure(figsize=(7, 6))
sns.heatmap(mat, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Validation Confusion Matrix', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Heartbeat Class')
plt.ylabel('Actual Heartbeat Class')
plt.show()


In [ ]:
# Measure the Accuracy Score and Classification Report
print('=== Validation Performance ===')
print(f'Accuracy score of the predictions: {accuracy_score(y_test, y_lbl) * 100:.2f}%')
print('\nDetailed Classification Report:')
print(classification_report(y_test, y_lbl, target_names=class_names))


> As we can see above CNN got accuracy of 96.6%, now we can predict the vaalues for our test dataset.

***

In [ ]:
# Load test dataset with multi-path resolution
test_candidates = [
    'data/mitbih_test.csv',
    '../data/mitbih_test.csv',
    'input/heartbeat/mitbih_test.csv',
    '../input/heartbeat/mitbih_test.csv'
]
test_path = next((p for p in test_candidates if os.path.exists(p)), 'data/mitbih_test.csv')
test_data = pd.read_csv(test_path, header=None)
test_df = pd.DataFrame(test_data)
print(f'Successfully loaded test dataset from: {test_path}')
print(f'Test Dataset Shape: {test_df.shape}')


In [ ]:
test_df.head()

In [ ]:
# Test dataset class distribution (Fixed for Seaborn 0.13+)
plt.figure(figsize=(9, 4))
sns.countplot(x=test_df[187], hue=test_df[187], palette='magma', legend=False)
plt.title('Test Dataset Class Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Heartbeat Class Label')
plt.ylabel('Sample Count')
plt.xticks(ticks=range(5), labels=class_names)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Plot random test sample waveform for each heartbeat class (Full 187 sample points)
fig, ax = plt.subplots(nrows=1, ncols=5, figsize=(25, 3))
for i in range(5):
    sample_df = test_df[test_df[187] == float(i)]
    if len(sample_df) > 0:
        waveform = sample_df.sample(1, random_state=42).iloc[0, :187].values
        ax[i].plot(waveform, color=colors[i], linewidth=2)
        ax[i].set_title(class_names[i], fontsize=12, fontweight='bold')
        ax[i].set_xlabel('Sample Index (0-186)')
        ax[i].set_ylabel('Normalized Voltage')
        ax[i].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Prepare test dataset tensors: (samples, channels, length)
test_X_raw = test_df.drop([187], axis=1).values
test_y_true = test_df[187].values.astype(int)

test_X = np.array(test_X_raw).reshape(test_X_raw.shape[0], 1, test_X_raw.shape[1])
test_X_t = torch.tensor(test_X, dtype=torch.float32)
eval_test_loader = DataLoader(TensorDataset(test_X_t), batch_size=128, shuffle=False)
print(f'Test feature shape: {test_X.shape}')


In [ ]:
# Run test inference
clf.eval()
test_preds = []
with torch.no_grad():
    for (batch_x,) in eval_test_loader:
        batch_x = batch_x.to(device)
        outputs = clf(batch_x)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())

test_lbl_y = np.array(test_preds)


In [ ]:
# Test Set Confusion Matrix Heatmap
mat_test = confusion_matrix(test_y_true, test_lbl_y)
plt.figure(figsize=(7, 6))
sns.heatmap(mat_test, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Test Dataset Confusion Matrix (1D-CNN Performance)', fontsize=12, fontweight='bold')
plt.xlabel('Predicted Heartbeat Class')
plt.ylabel('True Heartbeat Class')
plt.show()


In [ ]:
# Measure Test Accuracy Score and Comprehensive Report
print('=== Final Held-Out Test Evaluation ===')
print(f'Accuracy score of the predictions: {accuracy_score(test_y_true, test_lbl_y) * 100:.2f}%')
print('\nTest Classification Report:')
print(classification_report(test_y_true, test_lbl_y, target_names=class_names))


> And here our CNN got us 96.5% accuracy on test dataset